In [15]:
pip install mahjong

In [16]:
import re
from mahjong.hand_calculating.hand import HandCalculator
from mahjong.tile import TilesConverter
from mahjong.hand_calculating.hand_config import HandConfig, OptionalRules
from mahjong.meld import Meld

def point_count(
    input_str: str,
    allow_multi_yakuman: bool = True,
    enable_kuitan: bool = True,
    disable_aka_dora: bool = False
) -> list:

    calculator = HandCalculator()

    # -----------------------------------------
    # 1. 解析字串
    # -----------------------------------------
    parts = input_str.split('+')
    hand_str = parts[0]
    win_tile_str = None
    is_tsumo = True
    doras_str = ""
    extras_str = ""
    melds_str_list = []

    bakaze = 1
    jikaze = 2

    for p in parts[1:]:
        if p.startswith('d'):
            doras_str = p[1:]
        elif re.match(r'^[triwhk]*\d{0,2}$', p) and not re.search(r'[mpsz]', p):
            extras_str = p
        elif re.search(r'[mpsz]', p):
            if win_tile_str is None and len(p) <= 2:
                win_tile_str = p
                is_tsumo = False
            else:
                melds_str_list.append(p)

    wind_match = re.search(r'(\d{1,2})$', extras_str)
    if wind_match:
        w_str = wind_match.group(1)
        if len(w_str) == 1:
            bakaze = jikaze = int(w_str[0])
        elif len(w_str) == 2:
            bakaze = int(w_str[0])
            jikaze = int(w_str[1])

    wind_map = {1: 27, 2: 28, 3: 29, 4: 30}
    wind_names = {1: '東', 2: '南', 3: '西', 4: '北'}

    has_t = 't' in extras_str
    has_r = 'r' in extras_str
    has_i = 'i' in extras_str
    has_w = 'w' in extras_str
    has_h = 'h' in extras_str
    has_k = 'k' in extras_str

    # -----------------------------------------
    # 2. 全局唯一 136 ID 分配器
    # -----------------------------------------
    used_ids = set()
    def string_to_unique_136(s):
        if not s: return []
        try:
            raw_136 = TilesConverter.one_line_string_to_136_array(s, has_aka_dora=not disable_aka_dora)
        except Exception as e:
            raise ValueError(f"Tile Parse Error: {e}")

        res = []
        for t in raw_136:
            if t not in used_ids:
                used_ids.add(t)
                res.append(t)
            else:
                group_start = (t // 4) * 4
                for i in range(4):
                    new_t = group_start + i
                    if new_t not in used_ids:
                        used_ids.add(new_t)
                        res.append(new_t)
                        break
        return res

    # -----------------------------------------
    # 3. 轉換牌理並組裝
    # -----------------------------------------
    try:
        tiles_136 = string_to_unique_136(hand_str)
        if win_tile_str:
            win_tile_136 = string_to_unique_136(win_tile_str)[0]
            tiles_136.append(win_tile_136)
        else:
            win_tile_136 = tiles_136[-1]
    except Exception as e:
        return [False, [], 0, 0, 0, 0, f"Parse Error: ({str(e)})"]

    melds = []
    for m_str in melds_str_list:
        sub_melds = re.findall(r'([aAcC]?)(\d+[mpsz])', m_str)
        for prefix, sm in sub_melds:
            m_tiles = string_to_unique_136(sm)
            if len(m_tiles) not in [3, 4]:
                return [False, [], 0, 0, 0, 0, f"Parse Error: '{sm}' 格式錯誤 ({len(m_tiles)}張牌)。"]

            tiles_136.extend(m_tiles)
            is_closed = prefix.lower() in ['a', 'c']

            if len(m_tiles) == 3:
                if m_tiles[0] // 4 == m_tiles[1] // 4 == m_tiles[2] // 4:
                    melds.append(Meld(Meld.PON, m_tiles, True))
                else:
                    melds.append(Meld(Meld.CHI, m_tiles, True))
            elif len(m_tiles) == 4:
                melds.append(Meld(Meld.KAN, m_tiles, not is_closed))

    dora_indicators = string_to_unique_136(doras_str)

    # -----------------------------------------
    # 4. 規則與計算
    # -----------------------------------------
    config = HandConfig(
        is_tsumo=is_tsumo,
        is_riichi=has_r,
        is_ippatsu=has_i,
        is_daburu_riichi=has_w,
        is_haitei=has_h and is_tsumo,
        is_houtei=has_h and not is_tsumo,
        is_rinshan=has_k and is_tsumo,
        is_chankan=has_k and not is_tsumo,
        is_tenhou=has_t and jikaze == 1,
        is_chiihou=has_t and jikaze != 1,
        player_wind=wind_map.get(jikaze, 28),
        round_wind=wind_map.get(bakaze, 27),
        options=OptionalRules(
            has_open_tanyao=enable_kuitan,
            has_aka_dora=not disable_aka_dora,
            has_double_yakuman=allow_multi_yakuman
        )
    )

    try:
        result = calculator.estimate_hand_value(tiles_136, win_tile_136, melds, dora_indicators, config)
    except Exception as e:
        return [False, [], 0, 0, 0, 0, f"Calculate Exception: {str(e)}"]

    # -----------------------------------------
    # 5. 格式化輸出
    # -----------------------------------------
    if result.error is not None:
        return [False, [], 0, 0, 0, 0, f"Error: {result.error}"]

    yaku_list = [getattr(y, 'japanese', y.name) for y in result.yaku]

    # 點數分配處理
    pay_detail = ""
    if is_tsumo:
        main_pay = result.cost['main']
        add_pay = result.cost.get('additional', 0)
        if jikaze == 1: # 莊家自摸
            ten = main_pay * 3
            pay_detail = f" {main_pay}all"
        else: # 閒家自摸
            ten = main_pay + add_pay * 2
            pay_detail = f" ({main_pay}, {add_pay})"
    else:
        ten = result.cost['main']

    # 役滿倍數與封頂處理
    # 役滿倍數計算
    yakuman_yaku = [y for y in result.yaku if getattr(y, 'is_yakuman', False)]
    yakuman_count = sum(yaku.han_closed for yaku in yakuman_yaku) // 13 if yakuman_yaku else 0

    if not allow_multi_yakuman and yakuman_count > 1:
        yakuman_count = 1
        result.han = 13
        if jikaze == 1:
            ten, pay_detail = 48000, " 16000all"
        else:
            ten, pay_detail = 32000, " (16000, 8000)"

    agari_type = "自摸" if is_tsumo else "栄和"
    b_wind = wind_names.get(bakaze, '東')
    j_wind = wind_names.get(jikaze, '南')

    # 文字描述組合
    if yakuman_count > 1:
        base_text = f"{yakuman_count}倍役滿"
    elif yakuman_count == 1:
        base_text = "役滿"
    elif result.han >= 13:
        base_text = "累計役滿"
    else:
        base_text = f"{result.fu}符{result.han}飜"

    desc_text = f"({b_wind}場{j_wind}家){agari_type} {base_text} {ten}点{pay_detail if is_tsumo else ''}"

    return [True, yaku_list, yakuman_count, result.han, result.fu, ten, desc_text]

In [17]:
# ==========================================
# 測試區塊
# ==========================================
print("Test 1 (何意味):", point_count('5z5z+a1111za2222za3333za4444z+lk'))
print("Test 2 (純正九連):", point_count('1112345678999m+1m+d12s+ri13'))
print("Test 3 (字一色大三元單騎禁多倍役滿):", point_count('1z+1z+333z555z666z7777z+d12s+h14', allow_multi_yakuman=False))
print("Test 4 (大七星):", point_count('1122334455667z+7z+d12+14'))
print("Test 5 (綠一色):", point_count('2233448s+8s+666s666z'))
print("Test 6 (小三元三暗刻對對):", point_count('111p22s55666777z+2s+d5s8p+34'))
print("Test 7 (南南一氣):", point_count('12346789s77p222z5s+d3s+22'))
print("Test 8 (七對子):", point_count('11880p66s3377m33z+5p+d4p2z+ri22'))
print("Test 9 (國士無雙十三面禁多倍役滿):", point_count('19p19s19m1234567z+7z+d5s+22', allow_multi_yakuman=False))
print("Test 10 (華麗石上三年):", point_count('345s34077m34455p3p+d2s6m+wh12'))
print("Test 11 (詐和):", point_count('2233457778966m+d2s6m+wh12'))

Test 1 (何意味): [True, ['Suu Kantsu', 'Tsuu Iisou', 'Dai Suushii', 'Suu Ankou Tanki'], 6, 78, 160, 192000, '(東場南家)自摸 6倍役滿 192000点 (96000, 48000)']
Test 2 (純正九連): [True, ['Daburu Chuuren Poutou'], 2, 26, 40, 64000, '(東場西家)栄和 2倍役滿 64000点']
Test 3 (字一色大三元單騎禁多倍役滿): [True, ['Daisangen', 'Tsuu Iisou'], 1, 13, 60, 32000, '(東場北家)栄和 役滿 32000点']
Test 4 (大七星): [True, ['Tsuu Iisou'], 1, 13, 25, 32000, '(東場北家)栄和 役滿 32000点']
Test 5 (綠一色): [True, ['Ryuuiisou'], 1, 13, 30, 32000, '(東場南家)栄和 役滿 32000点']
Test 6 (小三元三暗刻對對): [True, ['Yakuhai (hatsu)', 'Yakuhai (chun)', 'Toitoi', 'San Ankou', 'Shou Sangen'], 0, 8, 60, 16000, '(西場北家)栄和 60符8飜 16000点']
Test 7 (南南一氣): [True, ['Menzen Tsumo', 'Yakuhai (seat wind south)', 'Yakuhai (round wind south)', 'Ittsu', 'Dora'], 0, 6, 30, 12000, '(南場南家)自摸 30符6飜 12000点 (6000, 3000)']
Test 8 (七對子): [True, ['Riichi', 'Ippatsu', 'Chiitoitsu', 'Dora', 'Aka Dora'], 0, 9, 25, 16000, '(南場南家)栄和 25符9飜 16000点']
Test 9 (國士無雙十三面禁多倍役滿): [True, ['Kokushi Musou Juusanmen Matchi'], 1, 13, 0,